In [1]:
import numpy as np
import itertools
from pyspark.sql import SparkSession
import time

In [2]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Ali'sSimHash") \
    .master("local[*]") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()


In [3]:
def read_fvecs(path):
    data = np.fromfile(path, dtype=np.int32)
    d = data[0]
    data = data.reshape(-1, d+1)
    return data[:, 1:].view(np.float32)

def read_ivecs(path):
    data = np.fromfile(path, dtype=np.int32)
    d = data[0]
    data = data.reshape(-1, d+1)
    return data[:, 1:]

In [12]:
def random_normal_matrix(m, d, seed = 42):
    return np.random.default_rng(seed).standard_normal((m, d)).astype(np.float32)

def compute_signature(q, hyperPlanesMatrix):

    results = hyperPlanesMatrix @ q.T
    sig = (results > 0).T
    return  sig
def signature_to_band(signature_1d, b):

    m = signature_1d.shape[0]
    r = m // b
    bands = []
    for idx in range(b):
        start = idx * r
        end = (idx + 1) * r
        band_bits = signature_1d[start:end].astype(np.uint8)
        if r <= 64:
            bit_str = "".join(band_bits.astype(str))
            val = int(bit_str, 2)
        else:
            bit_str = "".join(band_bits.astype(str))
            val = int(bit_str, 2)
        bands.append((idx, val))
    return bands
def emit_bands(item, b):
        doc_id, signature = item
        signatures = np.array(signature, dtype=bool)
        # doc_ids ia a list of doc ids in the same band bucket
        res = [((bi, bv), doc_id) for bi, bv in signature_to_band(signatures, b)]
        return res

def emit_pairs(item):
     _, doc_ids = item
     return [(min(a, b_), max(a, b_)) for a, b_ in itertools.combinations(doc_ids, 2)]

def k_to_cosine_threshold(k, m):
    theta = (k / m) * np.pi
    return float(np.cos(theta))

def evaluate_k(candidates, base, k, m):
    theoretical_threshold = k_to_cosine_threshold(k, m)
    good = 0
    total = len(candidates)

    for i, j in candidates:
        cos_sim = float(np.dot(base[i], base[j]))
        if(cos_sim >= theoretical_threshold):
            good += 1
    
    quality = 0.0

    if(good > 0):
        quality = good / total

    return {
        "k": k,
        "threshold": theoretical_threshold,
        "num_good": good,
        "num_candidates": total,
        "quality": quality

    }

In [5]:
def init_run_simhash(sparkSession, vectors, m, b, seed=42):
    num_dims = vectors.shape[1]
    hyperPlanes = random_normal_matrix(m, num_dims, seed)

    # compute signatures: (n_docs, m)
    signatures = compute_signature(vectors, hyperPlanes)
    print("signatures shape:", signatures.shape)  # (n_docs, m)

    # enumerate documents
    indexed = list(enumerate(signatures.tolist()))  # list of (doc_id, signature_list)

    rdd = sparkSession.sparkContext.parallelize(indexed, numSlices=6)

    band_rdd = rdd.flatMap(lambda x: emit_bands(x, b))

    # group by band key and keep buckets with at least 2 docs
    grouped = (band_rdd.groupByKey()
                        .mapValues(list)
                        .filter(lambda x: len(x[1]) >= 2))

    # for all pairs within each bucket
    candidate_pairs = grouped.flatMap(emit_pairs).distinct().collect()

    return set(candidate_pairs)

In [6]:
SIFT_paths = {
    "small": {
        "base": "./data/small/siftsmall_base.fvecs",
        "gt": "./data/small/siftsmall_groundtruth.ivecs",
        "queries": "./data/small/siftsmall_query.fvecs"
    }
}

N_VECTORS = 500
THRESHOLD = 0.9
SEED = 42


mVals = [32]
bVals = [4]

mode = "small"

base = read_fvecs(SIFT_paths[mode]["base"])
queries = read_fvecs(SIFT_paths[mode]["queries"])
groundTruts = read_ivecs(SIFT_paths[mode]["gt"])

base_normalized = base / np.linalg.norm(base, axis=1, keepdims=True)
queries_normalized = queries / np.linalg.norm(queries, axis=1, keepdims=True)

print(f"Loaded data successfully.\nShape: {base_normalized.shape}")

Loaded data successfully.
Shape: (10000, 128)


In [25]:
#spark = SparkSession.builder.appName("Ali'sSimHash").master("local[*]").config("spark.ui.showConsoleProgress", "false").getOrCreate()


for m in mVals:
    for b in bVals:
        if (m % b != 0):
            # illegal pair,
            continue
        
        r = m / b
        t0 = time.perf_counter()
        candidates = init_run_simhash(spark, base_normalized, m, b, seed=SEED)
        t0 = time.perf_counter() - t0
        print(t0)

    


signatures shape: (10000, 32)
93.93207480000092


In [16]:
ks = [1, 2, 3, 4,  5, 6, 8]

for k in ks:
    print(k_to_cosine_threshold(k, 64))

0.9987954562051724
0.9951847266721969
0.989176509964781
0.9807852804032304
0.970031253194544
0.9569403357322088
0.9238795325112867


In [30]:
res = evaluate_k(candidates, base_normalized, 10, 32)

In [31]:
res

{'k': 10,
 'threshold': 0.5555702330196023,
 'num_good': 4726861,
 'num_candidates': 7031752,
 'quality': 0.672216682272071}

In [24]:
for i, j in candidates:
    print(" ", i, " ", j)
    

  1271   4513
  3891   4042
  2117   5473
  4789   5192
  4486   5383
  9711   9825
  4508   9114
  1065   3856
  1682   8928
  963   3579
  7026   8366
  6974   8176
  985   7310
  1052   2116
  809   3112
  5150   9520
  1707   4262
  4260   8985
  0   499
  52   689
  898   1649
  1426   8184
  2391   4140
  7668   8772
  5880   7087
  2503   5135
  677   6376
  4247   7245
  3247   5818
  2016   8400
  7360   7838
  1386   1588
  2217   7932
  2284   2738
  319   5355
  267   5165
  1217   6505
  1165   6315
  2879   4079
  1284   1311
  3028   3421
  6176   9485
  2926   3144
  11   4421
  2705   7871
  3300   9212
  1896   5258
  2529   3673
  3308   9827
  5223   7123
  1742   4791
  3273   4356
  8550   8988
  8498   8798
  1218   9709
  1337   4705
  2992   8278
  5724   8802
  3141   7620
  3736   8961
  3320   3421
  6408   8680
  419   7814
  6468   9485
  434   2430
  1265   8774
  4532   9834
  3783   8026
  6152   7936
  4378   9367
  1076   2836
  3629   7559
  4527   8

KeyboardInterrupt: 